# V-JEPA 2 Demo Notebook

This tutorial provides an example of how to load the V-JEPA 2 model in vanilla PyTorch and HuggingFace, extract a video embedding, and then predict an action class. For more details about the paper and model weights, please see https://github.com/facebookresearch/vjepa2.

First, let's import the necessary libraries and load the necessary functions for this tutorial.

In [5]:
!pip install torch torchvision torchaudio
!pip install torchcodec

In [6]:
pip install -U git+https://github.com/huggingface/transformers


  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-mcmyh65g
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-mcmyh65g
  Resolved https://github.com/huggingface/transformers to commit b71de73468429eb02da18caa50e9b5200400a4ed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11553909 sha256=a09316c6d809ad90a7fe15c3ee0d8b4472f1fbff6ad7041f3ec80d89a5b9da34
  Stored in directory: /tmp/pip-ephem-wheel-cache-p2tvwz3q/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
# Clone the V-JEPA 2 repository if it doesn't exist
if not os.path.exists('vjepa2'):
    !git clone https://github.com/facebookresearch/vjepa2.git

# Add the cloned repository to Python's path
import sys
sys.path.insert(0, './vjepa2')

!pip install decord
import json
import os
import subprocess

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform


def get_video():
    vr = VideoReader("sample_video.mp4")
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, 128, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video


def forward_vjepa_video(model_hf, model_pt, hf_transform, pt_transform):
    # Run a sample inference with VJEPA
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        x_pt = pt_transform(video).cuda().unsqueeze(0)
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")
        # Extract the patch-wise features from the last layer
        out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf, out_patch_features_pt


def get_vjepa_video_classification_results(classifier, out_patch_features_pt):
    SOMETHING_SOMETHING_V2_CLASSES = json.load(open("ssv2_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features_pt)

    print(f"Classifier output shape: {out_classifier.shape}")

    print("Top 5 predicted class names:")
    top5_indices = out_classifier.topk(5).indices[0]
    top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage
    for idx, prob in zip(top5_indices, top5_probs):
        str_idx = str(idx.item())
        print(f"{SOMETHING_SOMETHING_V2_CLASSES[str_idx]} ({prob}%)")

    return

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Next, let's download a sample video to the local repository. If the video is already downloaded, the code will skip this step. Likewise, let's download a mapping for the action recognition classes used in Something-Something V2, so we can interpret the predicted action class from our model.

In [21]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.0 MB/s eta 0:00:00


In [7]:
!yt-dlp https://www.youtube.com/watch?v=iGi3Wsx9MZU -o french_toast.mp4

[youtube] Extracting URL: https://www.youtube.com/watch?v=iGi3Wsx9MZU
[youtube] iGi3Wsx9MZU: Downloading webpage
[youtube] iGi3Wsx9MZU: Downloading android vr player API JSON
[info] iGi3Wsx9MZU: Downloading 1 format(s): 137+251
[download] french_toast.mp4.mkv has already been downloaded


In [8]:
import os

sample_video_path = "french_toast.mp4.mkv"

if not os.path.exists(sample_video_path):
    raise FileNotFoundError("french_toast.mp4.mkv not found. Please download it first.")

print("Using video:", sample_video_path)

Using video: french_toast.mp4.mkv


In [10]:
#sample_video_path = "sample_video.mp4"
## Download the video if not yet downloaded to local path
#if not os.path.exists(sample_video_path):
 #   video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"
  #  command = ["wget", video_url, "-O", sample_video_path]
   # subprocess.run(command)
    #print("Downloading video")

# #Download SSV2 classes if not already present
#ssv2_classes_path = "ssv2_classes.json"
#if not os.path.exists(ssv2_classes_path):
 #   command = [
  #      "wget",
   #     "https://huggingface.co/datasets/huggingface/label-files/resolve/d79675f2d50a7b1ecf98923d42c30526a51818e2/"
    #    "something-something-v2-id2label.json",
     #   "-O",
      #  "ssv2_classes.json",
    #]
    #subprocess.run(command)
    #print("Downloading SSV2 classes")

Now, let's load the models in both vanilla Pytorch as well as through the HuggingFace API. Note that HuggingFace API will automatically load the weights through `from_pretrained()`, so there is no additional download required for HuggingFace.

To download the PyTorch model weights, use wget and specify your preferred target path. See the README for the model weight URLs.
E.g.
```
wget https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt -P YOUR_DIR
```
Then update `pt_model_path` with `YOUR_DIR/vitg-384.pt`. Also note that you have the option to use `torch.hub.load`.

In [9]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)
# Path to local PyTorch weights
#pt_model_path = "YOUR_MODEL_PATH"

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.

# Initialize the PyTorch model, load pretrained weights
#model_pt = vit_giant_xformers_rope(img_size=(img_size, img_size), num_frames=64)
#model_pt.cuda().eval()
#load_pretrained_vjepa_pt_weights(model_pt, pt_model_path)

### Can also use torch.hub to load the model
# model_pt, _ = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')
# model_pt.cuda().eval()

# Build PyTorch preprocessing transform
#pt_video_transform = build_pt_video_transform(img_size=img_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

Now we can run the encoder on the video to get the patch-wise features from the last layer of the encoder. To verify that the HuggingFace and PyTorch models are equivalent, we will compare the values of the features.

In [10]:
import numpy as np
import torch
from decord import VideoReader

def get_video(video_path="french_toast.mp4.mkv"):
    vr = VideoReader(video_path)
    # sample 64 frames uniformly from the video
    num_frames = len(vr)
    frame_idx = np.linspace(0, num_frames - 1, 64).astype(int)
    video = vr.get_batch(frame_idx).asnumpy()  # (T, H, W, C)
    video = torch.from_numpy(video).permute(0, 3, 1, 2)  # (T, C, H, W)
    return video

# ---- HF-only inference ----
with torch.inference_mode():
    video = get_video("french_toast.mp4.mkv")  # <-- change path if needed
    x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")

    out_patch_features_hf = model_hf.get_vision_features(x_hf)

print("HuggingFace output shape:", out_patch_features_hf.shape)

# Pool patch tokens -> one clip latent
z_clip = out_patch_features_hf.mean(dim=1)
print("Clip latent shape:", z_clip.shape)

torch.save(z_clip.cpu(), "z_clip.pt")
print("Saved z_clip.pt")

HuggingFace output shape: torch.Size([1, 18432, 1408])
Clip latent shape: torch.Size([1, 1408])
Saved z_clip.pt


In [11]:
import torch

torch.save(
    {
        "video_path": sample_video_path,
        "model": hf_model_name,
        "num_frames": 64,
        "pooling": "mean_over_tokens",
        "z_clip": z_clip.cpu(),  # (1, 1408)
    },
    "vjepa2_single_clip_latent.pt"
)
print("Saved vjepa2_single_clip_latent.pt")

Saved vjepa2_single_clip_latent.pt


In [12]:
import numpy as np
import torch
from decord import VideoReader

def extract_z_for_frame_range(
    video_path, start_f, end_f, model_hf, hf_transform,
    T=32, device="cuda", use_amp=True
):
    vr = VideoReader(video_path)
    start_f = max(0, start_f)
    end_f = min(len(vr) - 1, end_f)
    if end_f <= start_f:
        end_f = min(len(vr) - 1, start_f + 1)

    idx = np.linspace(start_f, end_f, T).astype(int)
    video = vr.get_batch(idx).asnumpy()
    video = torch.from_numpy(video).permute(0, 3, 1, 2)  # (T,C,H,W)

    # preprocess on CPU
    x = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to(device, non_blocking=True)

    with torch.inference_mode():
        ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()
        with ctx:
            tokens = model_hf.get_vision_features(x)       # (1, tokens, D)
            z = tokens.mean(dim=1)                         # (1, D)

    # move result to CPU immediately
    z_cpu = z.detach().cpu()

    # cleanup GPU memory
    del x, tokens, z
    torch.cuda.empty_cache()

    return z_cpu

def extract_uniform_segments(video_path, num_segments=8, T=32):
    vr = VideoReader(video_path)
    total = len(vr)
    bounds = np.linspace(0, total - 1, num_segments + 1).astype(int)

    zs = []
    for i in range(num_segments):
        z_i = extract_z_for_frame_range(video_path, bounds[i], bounds[i+1], model_hf, hf_transform, T=T)
        zs.append(z_i)
        print(f"segment {i}/{num_segments-1} done, z shape {tuple(z_i.shape)}")

    return torch.cat(zs, dim=0)

In [13]:
z_segments = extract_uniform_segments(sample_video_path, num_segments=8, T=32)
print("z_segments shape:", z_segments.shape)  # (8, 1408)

delta_z = z_segments[1:] - z_segments[:-1]
print("delta_z shape:", delta_z.shape)        # (7, 1408)

segment 0/7 done, z shape (1, 1408)
segment 1/7 done, z shape (1, 1408)
segment 2/7 done, z shape (1, 1408)
segment 3/7 done, z shape (1, 1408)
segment 4/7 done, z shape (1, 1408)
segment 5/7 done, z shape (1, 1408)
segment 6/7 done, z shape (1, 1408)
segment 7/7 done, z shape (1, 1408)
z_segments shape: torch.Size([8, 1408])
delta_z shape: torch.Size([7, 1408])


In [14]:
import torch

save_obj = {
    "video_path": sample_video_path,
    "model": hf_model_name,
    "num_segments": 8,
    "frames_per_segment": 32,          # because you used T=32
    "pooling": "mean_over_tokens",
    "z_segments": z_segments,          # (8, 1408)
    "delta_z": delta_z,                # (7, 1408)
}

torch.save(save_obj, "cross_task_one_video_vjepa2_latents.pt")
print("Saved cross_task_one_video_vjepa2_latents.pt")

Saved cross_task_one_video_vjepa2_latents.pt
